In [2]:
from __future__ import annotations

import csv
import sys
from collections import Counter, defaultdict
from pathlib import Path

# Nastavitve
ODSEK_COLUMN = 'odsek'
TOP_N = 50

# Poskusi samodejno najti pravi CSV ne glede na to, iz katere mape teče notebook.
BASE = Path.cwd().resolve()
DATA_DIR_CANDIDATES = [
    BASE / 'data',
    BASE.parent / 'data',
]
CSV_CANDIDATE_FILES = [
    'odseki.csv',
    'odseki_processed.csv',
    'odseki_nazivi.csv',
    'odseki_old.csv',
]


def resolve_csv_path() -> Path:
    for data_dir in DATA_DIR_CANDIDATES:
        for name in CSV_CANDIDATE_FILES:
            p = data_dir / name
            if p.exists():
                return p
    searched = [str(d / n) for d in DATA_DIR_CANDIDATES for n in CSV_CANDIDATE_FILES]
    raise FileNotFoundError(
        'CSV file not found. Tried:\n- ' + '\n- '.join(searched)
    )


CSV_PATH = resolve_csv_path()


def configure_csv_field_limit() -> None:
    limit = sys.maxsize
    while True:
        try:
            csv.field_size_limit(limit)
            break
        except OverflowError:
            limit //= 10


def analyze_odsek_values(csv_path: Path, odsek_column: str = 'odsek'):
    rows_meta = defaultdict(list)
    all_odsek_values = []

    with csv_path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            raise ValueError('CSV has no header row.')
        if odsek_column not in reader.fieldnames:
            raise ValueError(
                f"Column '{odsek_column}' not found. Available columns: {', '.join(reader.fieldnames)}"
            )

        for row_idx, row in enumerate(reader, start=2):  # header je vrstica 1
            odsek = (row.get(odsek_column) or '').strip()
            if not odsek:
                continue

            all_odsek_values.append(odsek)
            rows_meta[odsek].append({
                'line': row_idx,
                'ggo': (row.get('ggo') or '').strip(),
                'povrsina': (row.get('povrsina') or '').strip(),
            })

    counts = Counter(all_odsek_values)
    duplicates = {k: rows_meta[k] for k, c in counts.items() if c > 1}

    result = {
        'counts': counts,
        'duplicates': duplicates,
        'total_rows_with_odsek': len(all_odsek_values),
        'unique_odsek_count': len(counts),
        'duplicate_group_count': len(duplicates),
        'rows_in_duplicate_groups': sum(len(v) for v in duplicates.values()),
        'single_occurrence_count': sum(1 for c in counts.values() if c == 1),
    }
    return result


configure_csv_field_limit()

analysis = analyze_odsek_values(CSV_PATH, odsek_column=ODSEK_COLUMN)
dup = analysis['duplicates']

print(f'CSV: {CSV_PATH}')
print(f"Rows with non-empty '{ODSEK_COLUMN}': {analysis['total_rows_with_odsek']}")
print(f"Unique '{ODSEK_COLUMN}' values: {analysis['unique_odsek_count']}")
print(f"Duplicate '{ODSEK_COLUMN}' groups: {analysis['duplicate_group_count']}")
print(f"Rows involved in duplicate groups: {analysis['rows_in_duplicate_groups']}")
print(f"Single-occurrence '{ODSEK_COLUMN}' values: {analysis['single_occurrence_count']}")

print('\n# Top podvojene skupine')
sorted_items = sorted(dup.items(), key=lambda kv: (-len(kv[1]), kv[0]))
for odsek, rows in sorted_items[:TOP_N]:
    lines = ', '.join(str(r['line']) for r in rows)
    ggos = ', '.join(sorted({r['ggo'] for r in rows if r['ggo']}))
    print(f"- {odsek}: {len(rows)}x | lines: {lines} | ggo: {ggos or '-'}")

CSV: /home/tiln/Documents/BarkWatch_Arnes-Hackathon-2026_interface/data/odseki.csv
Rows with non-empty 'odsek': 53254
Unique 'odsek' values: 35688
Duplicate 'odsek' groups: 8703
Rows involved in duplicate groups: 26269
Single-occurrence 'odsek' values: 26985

# Top podvojene skupine
- 05013A: 9x | lines: 6922, 11031, 20761, 26876, 38127, 42029, 44727, 48573, 51393 | ggo: -
- 05074: 9x | lines: 7012, 20871, 26947, 29909, 38238, 42194, 44795, 48693, 51572 | ggo: -
- 01017A: 8x | lines: 6128, 10162, 19919, 26016, 28720, 37435, 47811, 50312 | ggo: -
- 01017B: 8x | lines: 6129, 10163, 19920, 26017, 28721, 37436, 47812, 50313 | ggo: -
- 03027A: 8x | lines: 6625, 20438, 22703, 26564, 29149, 41552, 48137, 50837 | ggo: -
- 03027B: 8x | lines: 6626, 20439, 22704, 26565, 29150, 41553, 48138, 50838 | ggo: -
- 03115: 8x | lines: 6728, 20583, 26614, 38029, 41712, 44455, 48282, 51042 | ggo: -
- 05013B: 8x | lines: 6923, 20762, 26877, 38128, 42030, 44728, 48574, 51394 | ggo: -
- 05039A: 8x | lines: 69

In [3]:
from collections import Counter, defaultdict

# Dodatna analiza: po katerih stolpcih se razlikujejo vrstice z istim odsek
INCLUDE_EMPTY_AS_VALUE = False   # če True, tudi '' šteje kot vrednost pri razlikah
DETAIL_ODSEK = '03101'           # nastavi None, če ne želiš podrobnosti za konkreten odsek
TOP_DIFF_COLUMNS = 20


def _norm(v):
    return (v or '').strip()


def _unique_values(values, include_empty=False):
    if include_empty:
        return sorted(set(values))
    return sorted({v for v in values if v != ''})


def analyze_duplicate_differences(csv_path: Path, odsek_column: str = 'odsek', include_empty=False):
    rows_by_odsek = defaultdict(list)

    with csv_path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            raise ValueError('CSV has no header row.')
        columns = reader.fieldnames

        for row_idx, row in enumerate(reader, start=2):
            odsek = _norm(row.get(odsek_column))
            if not odsek:
                continue

            clean_row = {k: _norm(v) for k, v in row.items()}
            clean_row['__line__'] = str(row_idx)
            rows_by_odsek[odsek].append(clean_row)

    duplicate_groups = {k: v for k, v in rows_by_odsek.items() if len(v) > 1}
    compare_columns = [c for c in columns if c != odsek_column]

    per_odsek_diff_cols = {}
    column_diff_counter = Counter()

    for odsek, rows in duplicate_groups.items():
        diff_cols = []
        for col in compare_columns:
            vals = [_norm(r.get(col)) for r in rows]
            uniq = _unique_values(vals, include_empty=include_empty)
            if len(uniq) > 1:
                diff_cols.append(col)
                column_diff_counter[col] += 1
        per_odsek_diff_cols[odsek] = set(diff_cols)

    return {
        'duplicate_groups': duplicate_groups,
        'per_odsek_diff_cols': per_odsek_diff_cols,
        'column_diff_counter': column_diff_counter,
        'compare_columns': compare_columns,
    }


analysis_diff = analyze_duplicate_differences(
    CSV_PATH,
    odsek_column=ODSEK_COLUMN,
    include_empty=INCLUDE_EMPTY_AS_VALUE,
)

per_odsek_diff_cols = analysis_diff['per_odsek_diff_cols']
column_diff_counter = analysis_diff['column_diff_counter']
duplicate_groups = analysis_diff['duplicate_groups']

print(f"\n# Analiza razlik za enak '{ODSEK_COLUMN}'")
print(f"Število podvojenih skupin: {len(duplicate_groups)}")

if len(duplicate_groups) == 0:
    print('Ni podvojenih skupin.')
else:
    no_diff_groups = sum(1 for cols in per_odsek_diff_cols.values() if len(cols) == 0)
    only_ggo_groups = sum(1 for cols in per_odsek_diff_cols.values() if cols == {'ggo'})

    print(f"Skupine brez razlik v ostalih stolpcih: {no_diff_groups}")
    print(f"Skupine, kjer se razlikuje samo 'ggo': {only_ggo_groups}")

    print(f"\nTop {TOP_DIFF_COLUMNS} stolpcev, ki se najpogosteje razlikujejo:")
    for col, count in column_diff_counter.most_common(TOP_DIFF_COLUMNS):
        print(f"- {col}: {count} skupin")

if DETAIL_ODSEK:
    rows = duplicate_groups.get(DETAIL_ODSEK)
    if not rows:
        print(f"\nOdsek {DETAIL_ODSEK} ni podvojen ali ne obstaja.")
    else:
        print(f"\n# Podrobnosti za odsek {DETAIL_ODSEK} (ponovitev: {len(rows)}x)")
        diff_cols = sorted(per_odsek_diff_cols.get(DETAIL_ODSEK, set()))
        print('Stolpci z razlikami:', ', '.join(diff_cols) if diff_cols else '(brez razlik)')

        show_cols = ['__line__', 'ggo'] + diff_cols
        # odstrani duplikate v vrstnem redu
        final_cols = []
        for c in show_cols:
            if c not in final_cols:
                final_cols.append(c)

        header = ' | '.join(final_cols)
        print(header)
        print('-' * len(header))
        for r in rows:
            print(' | '.join(_norm(r.get(c, '')) for c in final_cols))


# Analiza razlik za enak 'odsek'
Število podvojenih skupin: 8703
Skupine brez razlik v ostalih stolpcih: 0
Skupine, kjer se razlikuje samo 'ggo': 0

Top 20 stolpcev, ki se najpogosteje razlikujejo:
- ggo_naziv: 8703 skupin
- gge_naziv: 8703 skupin
- ke_naziv: 8703 skupin
- revir_naziv: 8703 skupin
- revirni: 8703 skupin
- eposta: 8703 skupin
- povrsina: 8701 skupin
- grt1_naziv: 8580 skupin
- lega_naziv: 8189 skupin
- krajime: 7927 skupin
- intgosp_naziv: 7486 skupin
- relief_naziv: 6911 skupin
- ohranjen_naziv: 6725 skupin
- pozar_naziv: 6338 skupin
- katgozd_naziv: 3576 skupin

# Podrobnosti za odsek 03101 (ponovitev: 5x)
Stolpci z razlikami: eposta, gge_naziv, ggo_naziv, grt1_naziv, intgosp_naziv, ke_naziv, krajime, lega_naziv, ohranjen_naziv, povrsina, pozar_naziv, relief_naziv, revir_naziv, revirni
__line__ | ggo | eposta | gge_naziv | ggo_naziv | grt1_naziv | intgosp_naziv | ke_naziv | krajime | lega_naziv | ohranjen_naziv | povrsina | pozar_naziv | relief_naziv | revir_naziv | 

In [4]:
# Preveri enoličnost kandidatnih ključev (ID kombinacij)
from collections import defaultdict


def uniqueness_report(csv_path: Path, key_cols: list[str]):
    counts = defaultdict(int)

    with csv_path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            key = tuple((row.get(c) or '').strip() for c in key_cols)
            # preskoči vrstice, kjer je ključ prazen
            if all(k == '' for k in key):
                continue
            counts[key] += 1

    total = len(counts)
    duplicate_keys = {k: v for k, v in counts.items() if v > 1}
    duplicate_key_count = len(duplicate_keys)
    duplicate_rows = sum(v for v in duplicate_keys.values())

    return {
        'key_cols': key_cols,
        'unique_key_values': total,
        'duplicate_key_count': duplicate_key_count,
        'duplicate_rows': duplicate_rows,
        'examples': sorted(duplicate_keys.items(), key=lambda kv: (-kv[1], kv[0]))[:10],
    }


candidate_keys = [
    ['odsek'],
    ['ggo', 'odsek'],
    ['ggo', 'odsek', 'katgozd'],
    ['ggo', 'odsek', 'geometry'],
]

print('\n# Enoličnost ključev')
for cols in candidate_keys:
    rep = uniqueness_report(CSV_PATH, cols)
    print(f"\nKljuč: {cols}")
    print(f"- unikatnih vrednosti ključa: {rep['unique_key_values']}")
    print(f"- podvojenih ključev: {rep['duplicate_key_count']}")
    print(f"- vrstic v podvojenih ključih: {rep['duplicate_rows']}")

    if rep['duplicate_key_count'] > 0:
        print('- primeri podvojenih ključev (do 10):')
        for key, n in rep['examples']:
            print(f"  {key} -> {n}x")


# Enoličnost ključev

Ključ: ['odsek']
- unikatnih vrednosti ključa: 35688
- podvojenih ključev: 8703
- vrstic v podvojenih ključih: 26269
- primeri podvojenih ključev (do 10):
  ('05013A',) -> 9x
  ('05074',) -> 9x
  ('01017A',) -> 8x
  ('01017B',) -> 8x
  ('03027A',) -> 8x
  ('03027B',) -> 8x
  ('03115',) -> 8x
  ('05013B',) -> 8x
  ('05039A',) -> 8x
  ('05039B',) -> 8x

Ključ: ['ggo', 'odsek']
- unikatnih vrednosti ključa: 35688
- podvojenih ključev: 8703
- vrstic v podvojenih ključih: 26269
- primeri podvojenih ključev (do 10):
  ('', '05013A') -> 9x
  ('', '05074') -> 9x
  ('', '01017A') -> 8x
  ('', '01017B') -> 8x
  ('', '03027A') -> 8x
  ('', '03027B') -> 8x
  ('', '03115') -> 8x
  ('', '05013B') -> 8x
  ('', '05039A') -> 8x
  ('', '05039B') -> 8x

Ključ: ['ggo', 'odsek', 'katgozd']
- unikatnih vrednosti ključa: 35688
- podvojenih ključev: 8703
- vrstic v podvojenih ključih: 26269
- primeri podvojenih ključev (do 10):
  ('', '05013A', '') -> 9x
  ('', '05074', '') -> 9x
  ('',